# 05 — Residual Feedforward Neural Network (ResFNN)

This notebook trains and evaluates the ResFNN — an improved version of the baseline FNN from `04_fnn.ipynb`.

## Key improvements over the baseline FNN

| Component | Baseline FNN | ResFNN |
|---|---|---|
| Architecture | Linear stack (128→64→32) | Residual blocks (256→128→64) |
| Normalisation | None | Batch Normalisation after each linear layer |
| Optimiser | Adam | AdamW (decoupled L2 weight decay) |
| LR schedule | Fixed | ReduceLROnPlateau (halves LR on plateau) |
| Loss | BCELoss | BCEWithLogitsLoss + pos_weight (class imbalance) |
| Patience | 10 epochs | 15 epochs |
| Max epochs | 100 | 200 |

## Architecture

```
Input(n)
  → Linear(256) → BN → ReLU → Dropout(0.3)      [input projection]
  → ResBlock(256→128): Linear→BN→ReLU→Drop→Linear→BN + skip
  → ResBlock(128→64):  Linear→BN→ReLU→Drop→Linear→BN + skip
  → Linear(1)                                    [output logit]
  → Sigmoid (inference only)
```

Results are appended to `outputs/results/metrics.csv`.  
Plots are saved to `outputs/figures/models/`.

## 0 · Imports

In [ ]:
import sys
sys.path.append('..')

import matplotlib.pyplot as plt

# Self modules
from src.data_loader import load_data
from src.preprocessing import preprocess
from src.models import split_data, get_X_y
from src.resfnn import train_resfnn, ResFNNWrapper
from src.evaluation import evaluate_model, plot_confusion_matrix, plot_roc_curve
from config import FEATURE_SETS, FEATURE_CONFIG, FIGURES_MODELS, PLOT_DPI

## 1 · Load & Split

Split is done on the **raw** DataFrame before preprocessing.  
This prevents any data leakage from imputation or encoding statistics.

In [ ]:
df_raw = load_data()

df_train_raw, df_test_raw = split_data(df_raw, test_seasons=[2022, 2023])

## 2 · ResFNN Feature Set Loop

Train the ResFNN on all feature sets and collect metrics.  
Training curve, confusion matrix, and ROC curve are saved for every feature set.

**Note:** This will take a few minutes on CPU — consider running on Google Colab for faster results.

In [ ]:
# Preprocess the largest feature set once; reuse column subsets to avoid redundant work.
LARGEST_FS = max(FEATURE_SETS, key=lambda k: len(FEATURE_SETS[k]))
_df_train_full = preprocess(df_train_raw, feature_set=FEATURE_SETS[LARGEST_FS], feature_config=FEATURE_CONFIG)
_df_test_full  = preprocess(df_test_raw,  feature_set=FEATURE_SETS[LARGEST_FS], feature_config=FEATURE_CONFIG)

for fs_name, fs_cols in FEATURE_SETS.items():

    print(f"\n{'='*55}")
    print(f" ResFNN | Feature set: {fs_name}  ({len(fs_cols)} features)")
    print(f"{'='*55}")

    # reuse precomputed preprocessing where possible
    is_subset = all(c in _df_train_full.columns for c in fs_cols)
    if is_subset:
        _df_train = _df_train_full[fs_cols + ["target"]]
        _df_test  = _df_test_full[fs_cols  + ["target"]]
    else:
        _df_train = preprocess(df_train_raw, feature_set=fs_cols, feature_config=FEATURE_CONFIG)
        _df_test  = preprocess(df_test_raw,  feature_set=fs_cols, feature_config=FEATURE_CONFIG)

    _X_train, _y_train = get_X_y(_df_train)
    _X_test,  _y_test  = get_X_y(_df_test)

    # train
    _model, _history, _scaler = train_resfnn(
        X_train=_X_train,
        y_train=_y_train,
        val_split=0.1,
        epochs=200,
        batch_size=512,
        lr=3e-4,
        patience=15,
        dropout=0.3,
        weight_decay=1e-2,
    )
    _wrapper = ResFNNWrapper(_model, scaler=_scaler)

    # evaluate + save to metrics.csv
    evaluate_model(_wrapper, _X_test, _y_test, "ResFNN", fs_name)

    # ── Sanity check ──────────────────────────────────────────────────────
    print(f"Train NaNs : {_df_train.isna().sum().sum()}")
    print(f"Test  NaNs : {_df_test.isna().sum().sum()}")
    print(f"Train shape: {_df_train.shape}")
    print(f"Test  shape: {_df_test.shape}")
    print(f"Target distribution (train):\n{_df_train['target'].value_counts(normalize=True).round(3)}")

    # ── Training curve ────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # loss
    axes[0].plot(_history["train_loss"], label="Train loss")
    axes[0].plot(_history["val_loss"],   label="Val loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("BCE Loss")
    axes[0].set_title(f"ResFNN Training Curve — {fs_name}")
    axes[0].legend()

    # learning rate — shows when the scheduler kicked in
    axes[1].plot(_history["lr"], color="darkorange")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Learning Rate")
    axes[1].set_title(f"LR Schedule — {fs_name}")
    axes[1].set_yscale("log")

    fig.tight_layout()
    FIGURES_MODELS.mkdir(parents=True, exist_ok=True)
    save_path = FIGURES_MODELS / f"resfnn_training_curve_{fs_name}.png"
    fig.savefig(save_path, dpi=PLOT_DPI)
    plt.show()
    print(f"Saved → {save_path}")

    # ── Confusion matrix ──────────────────────────────────────────────────
    plot_confusion_matrix(_wrapper, _X_test, _y_test, "ResFNN", fs_name)

    # ── ROC curve ─────────────────────────────────────────────────────────
    plot_roc_curve(
        models={"ResFNN": _wrapper},
        X_test=_X_test,
        y_test=_y_test,
        feature_set=fs_name,
    )

print("\nDone. ResFNN results saved to metrics.csv")

## 3 · Compare with Baseline

Load `metrics.csv` and compare ResFNN against the baseline FNN and XGBoost across all feature sets.

In [ ]:
from src.evaluation import compare_models
import pandas as pd

df_metrics = compare_models()

# filter to the models we care about for this comparison
df_compare = df_metrics[df_metrics["model"].isin(["FNN", "ResFNN", "XGBoost"])].copy()

# highlight best ROC-AUC per feature set
display(
    df_compare
    .sort_values(["feature_set", "roc_auc"], ascending=[True, False])
    .reset_index(drop=True)
    .style.highlight_max(subset=["roc_auc", "f1", "accuracy"], color="lightgreen")
)

In [ ]:
# Bar chart: ROC-AUC by model and feature set
import matplotlib.pyplot as plt
import numpy as np

feature_sets = df_compare["feature_set"].unique()
models       = df_compare["model"].unique()
x            = np.arange(len(feature_sets))
width        = 0.25

fig, ax = plt.subplots(figsize=(9, 5))

for i, model in enumerate(models):
    vals = [
        df_compare.loc[
            (df_compare["model"] == model) & (df_compare["feature_set"] == fs), "roc_auc"
        ].values[0]
        if len(df_compare.loc[
            (df_compare["model"] == model) & (df_compare["feature_set"] == fs)
        ]) > 0 else 0
        for fs in feature_sets
    ]
    ax.bar(x + i * width, vals, width, label=model)

ax.set_xlabel("Feature Set")
ax.set_ylabel("ROC-AUC")
ax.set_title("ROC-AUC Comparison: FNN vs ResFNN vs XGBoost")
ax.set_xticks(x + width)
ax.set_xticklabels(feature_sets)
ax.set_ylim(0.74, 0.82)
ax.legend()
ax.yaxis.grid(True, linestyle="--", alpha=0.7)
fig.tight_layout()

save_path = FIGURES_MODELS / "resfnn_vs_baseline_roc_auc.png"
fig.savefig(save_path, dpi=PLOT_DPI)
plt.show()
print(f"Saved → {save_path}")